# Prompt Engineering Seminar - Complete Tutorial

**Data Science Seminars - Seminar 5: Natural Language Processing**  
*June 11, 2025*

---

## Table of Contents
1. [Setup and Introduction](#setup)
2. [Prompt Engineering Basics](#prompt-basics)
3. [Advanced Prompt Engineering Techniques](#advanced-techniques)
4. [Practical Exercises](#exercises)

---

## 1. Setup and Introduction

### Install Required Libraries

In [51]:
# Install required packages (run this cell first)
!pip install openai python-dotenv pandas numpy matplotlib

Looking in indexes: https://mirrors.aliyun.com/pypi/simple/


### Import Libraries and Setup

In [52]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import time
from collections import Counter
import re
from typing import List, Dict, Any
import warnings
warnings.filterwarnings('ignore')

# For API management
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Set up plotting style
plt.style.use('default')

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


### API Setup (Secure Method)

**Important:** Create a `.env` file in your project directory with your API key:  
```
OPENAI_API_KEY=your_api_key_here
```

In [53]:
load_dotenv()


True

```
OPENAI_API_KEY=sk-yyeAa4WAdd3qEBQHqCKdiK0TVpsCGwYQAZO36kb3CbomTiJZ
BASE_URL=https://yunwu.ai/v1
```

In [54]:
# Load API key from environment
api_key = os.getenv('OPENAI_API_KEY')
base_url = os.getenv('BASE_URL')

if not api_key:
    print("⚠️ WARNING: No API key found!")
    print("Please create a .env file with your API key:")
    print("OPENAI_API_KEY=your_key_here")
    api_key = input("Or enter your API key here: ")

if not base_url:
    print("⚠️ WARNING: No base URL found!")
    print("Please create a .env file with your base URL:")
    print("BASE_URL=your_base_url_here")
    base_url = input("Or enter your base URL here: ")

# Initialize OpenAI client with the provided configuration
client = OpenAI(
    base_url=base_url,
    api_key=api_key,
    timeout=120
)


In [55]:
def test_api_connection():
    """Test if our API connection is working"""
    try:
        response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": "Say 'Hello, World!' in exactly 2 words."}
            ],
            max_tokens=10
        )
        
        result = response.choices[0].message.content
        print(f"✅ API Test Successful!")
        print(f"Response: {result}")
        return True

    except Exception as e:
        print(f"❌ API Test Failed: {e}")
        return False

# Test the connection
api_working = test_api_connection()

✅ API Test Successful!
Response: Hello, World!


## 2. Prompt Engineering Basics

### Understanding Prompt Components

In [56]:
def explore_prompt_components():
    """Explore different components of effective prompts"""
    
    print("🎯 PROMPT COMPONENTS ANALYSIS")
    print("="*50)
    
    # Example prompt with all components
    example_prompt = """
# Identity
You are a helpful assistant that labels product reviews as Positive, Negative, or Neutral.

# Instructions
* Only output a single word in your response
* Your response should be one of: "Positive", "Negative", or "Neutral"
* Focus on the overall sentiment expressed

# Examples
Review: "I absolutely love this product!"
Sentiment: Positive

Review: "It's okay, nothing special."
Sentiment: Neutral

Review: "Terrible quality, waste of money."
Sentiment: Negative

# Task
Review: "The headphones sound great but are uncomfortable."
Sentiment:
"""
    
    components = {
        'Identity/Role': 'Defines who the AI should be',
        'Instructions': 'Clear, specific directions',
        'Examples': 'Few-shot learning demonstrations',
        'Context': 'Background information',
        'Task': 'The actual work to be done',
        'Output Format': 'How the response should be structured'
    }
    
    print("📋 PROMPT COMPONENTS:")
    for component, description in components.items():
        print(f"   {component}: {description}")
    
    print(f"\n📝 EXAMPLE PROMPT:")
    print(example_prompt)
    
    return example_prompt

example_prompt = explore_prompt_components()

🎯 PROMPT COMPONENTS ANALYSIS
📋 PROMPT COMPONENTS:
   Identity/Role: Defines who the AI should be
   Instructions: Clear, specific directions
   Examples: Few-shot learning demonstrations
   Context: Background information
   Task: The actual work to be done
   Output Format: How the response should be structured

📝 EXAMPLE PROMPT:

# Identity
You are a helpful assistant that labels product reviews as Positive, Negative, or Neutral.

# Instructions
* Only output a single word in your response
* Your response should be one of: "Positive", "Negative", or "Neutral"
* Focus on the overall sentiment expressed

# Examples
Review: "I absolutely love this product!"
Sentiment: Positive

Review: "It's okay, nothing special."
Sentiment: Neutral

Review: "Terrible quality, waste of money."
Sentiment: Negative

# Task
Review: "The headphones sound great but are uncomfortable."
Sentiment:



### Basic Prompt Testing

In [57]:
def test_basic_prompts():
    """Test different prompt approaches"""
    
    if not api_working:
        print("⚠️ Skipping API tests - connection not available")
        return
    
    print("🧪 PROMPT TESTING LABORATORY")
    print("="*50)
    
    # Test data
    test_review = "The product works well but delivery was slow."
    
    # Different prompt approaches
    prompts = {
        "Simple": f"Is this review positive or negative? '{test_review}'",
        
        "Structured": f"""Classify this product review sentiment.
Review: {test_review}
Sentiment(Positive/Negative/Neutral):""",
        
        "With_Examples": f"""Classify product review sentiment as Positive, Negative, or Neutral.

Examples:
"Amazing product!" → Positive
"It's okay" → Neutral  
"Terrible" → Negative

Review: {test_review}
Sentiment:""",
        
        "Role_Based": f"""You are an expert sentiment analyzer for e-commerce reviews.
Classify this review as Positive, Negative, or Neutral.
Only respond with the classification.

Review: {test_review}
Classification:"""
    }
    
    results = {}
    
    for prompt_type, prompt in prompts.items():
        try:
            print(f"\n🔍 Testing {prompt_type} prompt...")
            
            response = client.chat.completions.create(
                model="gpt-3.5-turbo",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.1
            )
            
            result = response.choices[0].message.content.strip()
            results[prompt_type] = result
            print(f"   Result: {result}")
            
            # Small delay to be respectful to API
            time.sleep(1)
            
        except Exception as e:
            print(f"   Error: {e}")
            results[prompt_type] = "Error"
    
    return results

prompt_results = test_basic_prompts()

🧪 PROMPT TESTING LABORATORY

🔍 Testing Simple prompt...
   Result: The review is mixed. It has a positive aspect ("the product works well") but also includes a negative aspect ("delivery was slow"). Overall, it suggests satisfaction with the product but dissatisfaction with the delivery experience.

🔍 Testing Structured prompt...
   Result: Sentiment: Neutral

🔍 Testing With_Examples prompt...
   Result: Neutral

🔍 Testing Role_Based prompt...
   Result: Neutral


## 3. Advanced Prompt Engineering Techniques

### Chain-of-Thought Prompting


In [58]:
def demonstrate_chain_of_thought():
    """Demonstrate Chain-of-Thought prompting technique"""
    
    print("🧠 CHAIN-OF-THOUGHT PROMPTING")
    print("="*50)
    
    # Math problem for demonstration
    problem = "Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many tennis balls does he have now?"
    
    # Standard prompt
    standard_prompt = f"{problem}"
    
    # Chain-of-thought prompt
    cot_prompt = f"""{problem}
    
Let's think step by step:"""
    
    # Few-shot chain-of-thought
    few_shot_cot = f"""Q: Janet has 3 apples. She buys 2 more apples. How many apples does she have now?
A: Janet started with 3 apples. She bought 2 more apples. 3 + 2 = 5. The answer is 5.

Q: {problem}
A:"""
    
    prompts = {
        "Standard": standard_prompt,
        "Zero-shot CoT": cot_prompt,
        "Few-shot CoT": few_shot_cot
    }
    
    print("📚 DIFFERENT PROMPTING APPROACHES:")
    for approach, prompt in prompts.items():
        print(f"\n{approach}:")
        print(f"'{prompt[:100]}...'") if len(prompt) > 100 else print(f"'{prompt}'")
    
    if api_working:
        print("\n🧪 TESTING APPROACHES:")
        results = {}
        
        for approach, prompt in prompts.items():
            try:
                response = client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.1
                )
                
                result = response.choices[0].message.content.strip()
                results[approach] = result
                print(f"\n{approach} Result:")
                print(f"{result}")
                
                time.sleep(1)
                
            except Exception as e:
                print(f"\n{approach} Error: {e}")
        
        return results
    else:
        print("\n⚠️ API not available for testing")
        return prompts

cot_results = demonstrate_chain_of_thought()

🧠 CHAIN-OF-THOUGHT PROMPTING
📚 DIFFERENT PROMPTING APPROACHES:

Standard:
'Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many...'

Zero-shot CoT:
'Roger has 5 tennis balls. He buys 2 more cans of tennis balls. Each can has 3 tennis balls. How many...'

Few-shot CoT:
'Q: Janet has 3 apples. She buys 2 more apples. How many apples does she have now?
A: Janet started w...'

🧪 TESTING APPROACHES:

Standard Result:
Roger starts with 5 tennis balls. He buys 2 cans of tennis balls, and each can contains 3 tennis balls. 

First, we calculate the total number of tennis balls in the cans:

\[
2 \text{ cans} \times 3 \text{ tennis balls per can} = 6 \text{ tennis balls}
\]

Now, we add the tennis balls he already has to the tennis balls from the cans:

\[
5 \text{ tennis balls} + 6 \text{ tennis balls} = 11 \text{ tennis balls}
\]

Therefore, Roger now has a total of 11 tennis balls.

Zero-shot CoT Result:
Sure! Let's break it down step by step:



## 4. Practical Exercises

### Exercise 1: Text Summarization

In [59]:
def exercise_text_summarization():
    """Practice text summarization with different prompt strategies"""
    
    print("📝 EXERCISE 1: TEXT SUMMARIZATION")
    print("="*50)
    
    # Sample article
    article = """Artificial Intelligence (AI) has revolutionized numerous industries over the past decade. From healthcare to finance, AI applications have demonstrated remarkable capabilities in pattern recognition, decision-making, and automation. In healthcare, AI systems can analyze medical images with accuracy that often surpasses human radiologists, helping to detect diseases earlier and more reliably. In the financial sector, AI algorithms are used for fraud detection, algorithmic trading, and risk assessment, processing vast amounts of data in real-time to make split-second decisions.
    
However, the rapid advancement of AI also brings challenges. Concerns about job displacement, privacy, and algorithmic bias have sparked important discussions about the responsible development and deployment of AI systems. Many experts argue that while AI will automate certain tasks, it will also create new job opportunities that require different skill sets. The key is ensuring that workers are retrained and educated to adapt to this changing landscape.
    
Looking forward, the integration of AI into daily life is expected to accelerate. Smart homes, autonomous vehicles, and personalized digital assistants are just the beginning. As AI becomes more sophisticated, it will likely play an increasingly important role in addressing global challenges such as climate change, disease prevention, and resource optimization."""

    print(f"📄 ORIGINAL ARTICLE ({len(article.split())} words):")
    print(article[:200] + "...")
    
    # Different summarization prompts
    summary_prompts = {
        "Basic": f"Summarize this article: {article}",
        
        "Structured": f"""Please provide a concise summary of the following article in 3-4 sentences.
        
Article: {article}
        
Summary:""",
        
        "Bullet_Points": f"""Summarize this article in bullet points, highlighting the key information:
        
Article: {article}
        
Key Points:""",
        
        "Executive_Summary": f"""You are a business analyst. Create an executive summary of this article for senior management.
Focus on implications and key takeaways.
        
Article: {article}
        
Executive Summary:"""
    }
    
    if api_working:
        print("\n🧪 TESTING DIFFERENT SUMMARIZATION APPROACHES:")
        summaries = {}
        
        for approach, prompt in summary_prompts.items():
            try:
                response = client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.3
                )
                
                summary = response.choices[0].message.content.strip()
                summaries[approach] = summary
                word_count = len(summary.split())
                
                print(f"\n{approach} Summary ({word_count} words):")
                print(summary)
                
                time.sleep(1)
                
            except Exception as e:
                print(f"\n{approach} Error: {e}")
        
        return summaries
    else:
        print("\n⚠️ API not available - showing prompt templates only")
        return summary_prompts

exercise_text_summarization()

📝 EXERCISE 1: TEXT SUMMARIZATION
📄 ORIGINAL ARTICLE (196 words):
Artificial Intelligence (AI) has revolutionized numerous industries over the past decade. From healthcare to finance, AI applications have demonstrated remarkable capabilities in pattern recognition, ...

🧪 TESTING DIFFERENT SUMMARIZATION APPROACHES:

Basic Summary (112 words):
The article discusses the transformative impact of Artificial Intelligence (AI) across various industries, particularly in healthcare and finance. AI excels in tasks like medical image analysis and real-time data processing for fraud detection and risk assessment. However, the rapid growth of AI raises concerns about job displacement, privacy issues, and algorithmic bias, prompting discussions on responsible AI development. Experts suggest that while AI may automate some jobs, it will also create new opportunities that require different skills, emphasizing the need for worker retraining. Looking ahead, AI's integration into everyday life is expecte

{'Basic': "The article discusses the transformative impact of Artificial Intelligence (AI) across various industries, particularly in healthcare and finance. AI excels in tasks like medical image analysis and real-time data processing for fraud detection and risk assessment. However, the rapid growth of AI raises concerns about job displacement, privacy issues, and algorithmic bias, prompting discussions on responsible AI development. Experts suggest that while AI may automate some jobs, it will also create new opportunities that require different skills, emphasizing the need for worker retraining. Looking ahead, AI's integration into everyday life is expected to increase, with potential applications in smart homes, autonomous vehicles, and addressing global challenges like climate change and disease prevention.",
 'Structured': 'Artificial Intelligence (AI) has significantly transformed various industries, particularly healthcare and finance, by enhancing capabilities in pattern recog

### Exercise 2: Information Extraction

In [ ]:
def exercise_information_extraction():
    """Practice extracting structured information from unstructured text"""
    
    print("🔍 EXERCISE 2: INFORMATION EXTRACTION")
    print("="*50)
    
    # Sample business email
    email_text = """Subject: Q4 2024 Sales Meeting - Action Items
    
From: Sarah Johnson <sarah.johnson@techcorp.com>
To: Marketing Team <marketing@techcorp.com>
Date: December 15, 2024
    
Hi Team,
    
Following our quarterly sales review meeting yesterday, here are the key action items:
    
1. Launch the new product campaign by January 15, 2025 (Owner: Mike Chen)
2. Increase social media budget by 25% for Q1 2025 (Owner: Lisa Wang)
3. Schedule client feedback sessions with top 5 customers (Owner: David Smith, Deadline: January 31, 2025)
4. Update the pricing strategy document (Owner: Jennifer Lee, Deadline: December 30, 2024)
    
Our Q4 revenue target was $2.5M and we achieved $2.8M, which is a 12% increase over our goal.
    
Next meeting scheduled for January 10, 2025 at 2:00 PM EST.
    
Best regards,
Sarah Johnson
Sales Director
TechCorp Inc.
Phone: (555) 123-4567"""
    
    print("📧 SAMPLE EMAIL:")
    print(email_text[:300] + "...")
    
    # Different extraction prompts
    extraction_prompts = {
        "Action_Items": f"""Extract all action items from this email including owner and deadline if mentioned:
        
{email_text}
        
Action Items:""",
        
        "Key_Metrics": f"""Extract all numerical metrics and KPIs mentioned in this email:
        
{email_text}
        
Metrics:""",
        
        "Contact_Info": f"""Extract all contact information (names, emails, phone numbers) from this email:
        
{email_text}
        
Contact Information:""",
        
        "JSON_Format": f"""Extract key information from this email and format as JSON with the following structure:
{{"sender": "", "date": "", "action_items": [], "metrics": {{}}, "next_meeting": ""}}
        
Email: {email_text}
        
JSON Output:"""
    }
    
    if api_working:
        print("\n🧪 TESTING DIFFERENT EXTRACTION APPROACHES:")
        extractions = {}
        
        for approach, prompt in extraction_prompts.items():
            try:
                response = client.chat.completions.create(
                    model="gpt-3.5-turbo",
                    messages=[{"role": "user", "content": prompt}],
                    temperature=0.1
                )
                
                extraction = response.choices[0].message.content.strip()
                extractions[approach] = extraction
                
                print(f"\n{approach.replace('_', ' ')} Extraction:")
                print(extraction)
                
                time.sleep(1)
                
            except Exception as e:
                print(f"\n{approach} Error: {e}")
        
        return extractions
    else:
        print("\n⚠️ API not available - showing prompt templates only")
        return extraction_prompts

extraction_results = exercise_information_extraction()

🔍 EXERCISE 2: INFORMATION EXTRACTION
📧 SAMPLE EMAIL:
Subject: Q4 2024 Sales Meeting - Action Items
    
From: Sarah Johnson <sarah.johnson@techcorp.com>
To: Marketing Team <marketing@techcorp.com>
Date: December 15, 2024
    
Hi Team,
    
Following our quarterly sales review meeting yesterday, here are the key action items:
    
1. Launch the new pro...

🧪 TESTING DIFFERENT EXTRACTION APPROACHES:

Action Items Extraction:
Here are the extracted action items from the email:

1. **Action Item:** Launch the new product campaign  
   **Owner:** Mike Chen  
   **Deadline:** January 15, 2025  

2. **Action Item:** Increase social media budget by 25% for Q1 2025  
   **Owner:** Lisa Wang  
   **Deadline:** Not mentioned  

3. **Action Item:** Schedule client feedback sessions with top 5 customers  
   **Owner:** David Smith  
   **Deadline:** January 31, 2025  

4. **Action Item:** Update the pricing strategy document  
   **Owner:** Jennifer Lee  
   **Deadline:** December 30, 2024

Key Met